In [1]:
from data import api
from data.transformations import plot_acf_pacf
from models.sarimax import sarimax
import polars as pl
import plotly.express as px
import geopandas as gpd
import pandas as pd
from models.transformations import influence
import numpy as np

In [ ]:
# get panel data
df = api.get_panel_data()
df

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt

from models.CAR.car import (
    prepare_panel_for_st_car,
    build_st_car_nb_model_icar,      # use NB version instead of Poisson
    fit_st_car_model,
    diagnostics_mcmc,
    diagnostics_ppc_residuals,
    diagnostics_spatial_temporal,
    forecast_st_car,
)

# df = api.get_panel_data()  # your Polars panel
outcome = "NUMBER_OF_HOMICIDIO"
cols = [f"S0{i}" for i in range(1, 9)]
components = 4

df_pd, y, X_stdzd, meta = prepare_panel_for_st_car(df, outcome=outcome)

meta


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

correlation_matrix = pd.concat([pd.DataFrame(y, columns=[outcome]), pd.DataFrame(X_stdzd, columns=[f'PC_{i}' for i in range(1, meta['pca']['n_components'] + 1)]+['population'])], axis=1).corr()

plt.figure(figsize=(8, 6)) # Adjust figure size for better readability
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix')
plt.show()


In [ ]:
from data.mappings import SECTOR_ID
# 4. Extract loadings for the first principal component
pc1_loadings = meta['pca']['components'][2]

# 5. Create a horizontal bar plot of PC1 loadings
plt.figure(figsize=(10, 6))
plt.barh([SECTOR_ID[col] for col in meta['pca']['sector_cols']], pc1_loadings, color='skyblue')
plt.xlabel('Loading Value')
plt.ylabel('Feature')
plt.title('Principal Component 3 Loadings (Weights)')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15, 10)) # Adjust figure size for better readability
sns.heatmap(meta['W_raw'], , annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Adjacency Matrix')
plt.show()

In [ ]:
from models.CAR.car import (
    prepare_panel_for_st_car,
    build_st_car_nb_model_icar,      # use NB version instead of Poisson
    fit_st_car_model,
    diagnostics_mcmc,
    diagnostics_ppc_residuals,
    diagnostics_spatial_temporal,
    forecast_st_car,
)

# df = api.get_panel_data()  # your Polars panel
outcome = "NUMBER_OF_HOMICIDIO"

df_pd, y, X_stdzd, meta = prepare_panel_for_st_car(df, outcome=outcome)

# Build NB-CAR instead of Poisson-CAR
model = build_st_car_nb_model_icar(df_pd, y, X_stdzd, meta)
trace, idata = fit_st_car_model(model)

# Same diagnostics as before
diagnostics_mcmc(idata)
resid_pearson, mu_hat, dq_resid = diagnostics_ppc_residuals(model, idata, df_pd, y)
diagnostics_spatial_temporal(resid_pearson, df_pd, meta)

# Forecast
df_forecast = forecast_st_car(trace, df_pd, meta, horizon_months=12)
print(df_forecast.head())


In [ ]:
from models.CAR import plots
from models.CAR import car

# Example: using PPC-based Pearson residuals
resid_pearson, mu_hat, var_hat = car.compute_residuals_pearson(model, idata, y)

gdf_car_resid = plots.spatial_residual_sum_nbcar(
    df_pd=df_pd,
    resid=resid_pearson,          # or raw residuals y - mu_hat if you prefer
    geojson_path="data/colombia_departments.geojson",
)

In [ ]:
from models.CAR.car import car_in_sample_cv, rolling_cv_car, nb_car_gof

# NB-CAR
metrics_in_nb, resid_nb, mu_nb = car_in_sample_cv(model, idata, df_pd, y)
cv_out_nb = rolling_cv_car(df, "NUMBER_OF_HOMICIDIO",
                           model_builder=build_st_car_nb_model_icar,
                           forecast_func=forecast_st_car)

cv_out_nb

In [ ]:
from models.CAR.car import nb_car_gof
gof_nb = nb_car_gof(model, idata, y, var_name="y_like")

In [24]:
def assess_spatial_nb_icar_adequacy(
    model,
    idata,
    df_pd: pd.DataFrame,
    y: np.ndarray,
    meta: dict,
    outcome_name: str,
    do_cv: bool = False,
    df_polars_original: pl.DataFrame | None = None,
):
    """
    Run a suite of model adequacy checks for the Spatial NB-ICAR model:
      - MCMC diagnostics (Rhat, ESS, trace plots, energy)
      - Posterior predictive residuals (Pearson + Dunn–Smyth)
      - Spatial & temporal diagnostics (Moran's I, ACF)
      - Posterior predictive GOF statistics (chi-square, zeros)
      - Optional: in-sample CV-style metrics (RMSE, MAE, coverage)

    Parameters
    ----------
    model : pm.Model
        Fitted PyMC model, e.g. from build_st_car_nb_model_icar.
    idata : arviz.InferenceData
        MCMC output from fit_st_car_model.
    df_pd : pandas.DataFrame
        Panel used in the model (output of prepare_panel_for_st_car).
    y : np.ndarray
        Outcome vector (1D) aligned with df_pd.
    meta : dict
        Metadata dictionary from prepare_panel_for_st_car.
    outcome_name : str
        Name of the outcome column (for reporting).
    do_cv : bool, default False
        If True, run in-sample CV-style evaluation using car_in_sample_cv.
    df_polars_original : pl.DataFrame or None
        Required if do_cv=True; this is the full original Polars panel
        in the same format used to fit the model.
    """

    print("\n================= MCMC DIAGNOSTICS =================")
    diagnostics_mcmc(idata)

    print("\n================= POSTERIOR PREDICTIVE CHECKS =================")
    resid_pearson, mu_hat, dq_resid = diagnostics_ppc_residuals(
        model, idata, df_pd, y, var_name="y_like"
    )

    print("\n================= SPATIAL & TEMPORAL DIAGNOSTICS =================")
    diagnostics_spatial_temporal(resid_pearson, df_pd, meta, max_lag=24)

    print("\n================= POSTERIOR PREDICTIVE GOF (DISCREPANCIES) =================")
    gof_results = nb_car_gof(
        model,
        idata,
        y_obs=y,
        var_name="y_like",
        n_ppc_samples=500,
    )
    print("GOF results dictionary:", gof_results)

    if do_cv:
        if df_polars_original is None:
            raise ValueError("df_polars_original must be provided when do_cv=True.")

        print("\n================= IN-SAMPLE CV-STYLE METRICS =================")
        metrics, resid_cv, mu_hat_cv = car_in_sample_cv(
            model, idata, df_pd, y, var_name="y_like"
        )
        print(f"In-sample metrics for {outcome_name}:")
        for k, v in metrics.items():
            print(f"  {k}: {v:.4f}")

    print("\n==== Adequacy summary ====")
    print("Inspect:")
    print("  - Rhat/ESS from MCMC diagnostics")
    print("  - Trace plots / energy plots for mixing")
    print("  - Hist / scatter / QQ plots from PPC residuals")
    print("  - Moran's I and temporal ACF of residuals")
    print("  - Chi-square and zero-count p_B from nb_car_gof")
    if do_cv:
        print("  - RMSE/MAE/coverage from in-sample CV metrics")


In [ ]:
assess_spatial_nb_icar_adequacy(
        model=model,
        idata=idata,
        df_pd=df_pd,
        y=y,
        meta=meta,
        outcome_name=outcome,
        do_cv=False,              # flip to True if you want in-sample CV metrics
        df_polars_original=df,    # needed only if do_cv=True
    )

In [35]:
import numpy as np
import pandas as pd

def extract_all_parameters(idata, meta, ci=0.95):
    """
    Extract all NB–ICAR parameters into one long DataFrame:

      - alpha, tau_s, sigma_t, alpha_nb
      - beta[k] (covariates / PCs + POPULATION)
      - u[i]    (spatial dept effects)
      - v[t]    (time trend)
      - gamma[m] (monthly seasonality)

    Returns
    -------
    df_params : DataFrame with columns:
      ["param", "index", "mean", "ci_lower", "ci_upper"]
    """

    rows = []
    lower_q = (1.0 - ci) / 2.0
    upper_q = 1.0 - lower_q

    # ---------- 1. Helper for scalars ----------
    def add_scalar(name):
        arr = idata.posterior[name]        # xarray: (chain, draw)
        vals = arr.values.reshape(-1)      # all post-warmup draws
        mean = vals.mean()
        lo, hi = np.quantile(vals, [lower_q, upper_q])
        rows.append(dict(
            param=name,
            index=None,
            mean=float(mean),
            ci_lower=float(lo),
            ci_upper=float(hi),
        ))

    # scalars
    for p in ["alpha", "tau_s", "sigma_t", "alpha_nb"]:
        if p in idata.posterior:
            add_scalar(p)

    # ---------- 2. Helper for vectors (last dim is the index) ----------
    def add_vector(name, labels, group_name=None):
        """
        name: var in idata.posterior (e.g. "beta", "u", "v", "gamma")
        labels: list of labels for last dimension
        group_name: name to use in 'param' column (default = name)
        """
        if group_name is None:
            group_name = name

        arr = idata.posterior[name].values   # shape (chain, draw, D)
        chain, draw, D = arr.shape
        samples = arr.reshape(chain * draw, D)  # (S, D)

        means = samples.mean(axis=0)
        qs = np.quantile(samples, [lower_q, upper_q], axis=0)  # (2, D)

        for j, lab in enumerate(labels):
            rows.append(dict(
                param=group_name,
                index=lab,
                mean=float(means[j]),
                ci_lower=float(qs[0, j]),
                ci_upper=float(qs[1, j]),
            ))

    # beta_k: regression coefficients
    if "beta" in idata.posterior:
        add_vector(
            name="beta",
            labels=meta["X_cols"],      # e.g. ["PC1", ..., "POPULATION"]
            group_name="beta",
        )

    # u_i: spatial dept effects
    if "u" in idata.posterior:
        dept_labels = [int(d) for d in meta["dept_codes"]]
        add_vector(
            name="u",
            labels=dept_labels,
            group_name="u",
        )

    # v_t: temporal trend
    if "v" in idata.posterior:
        time_labels = [str(t) for t in meta["time_vals"]]
        add_vector(
            name="v",
            labels=time_labels,
            group_name="v",
        )

    # gamma_m: month effects (assumed last dim = months)
    if "gamma" in idata.posterior:
        n_month = idata.posterior["gamma"].values.shape[-1]
        month_labels = list(range(1, n_month + 1))   # 1..12
        add_vector(
            name="gamma",
            labels=month_labels,
            group_name="gamma_month",
        )

    df_params = pd.DataFrame(rows)[
        ["param", "index", "mean", "ci_lower", "ci_upper"]
    ]
    return df_params


In [ ]:
df_params = extract_all_parameters(idata, meta)
print(df_params.head())

# if you want to check consistency with az.summary:
import arviz as az
print(az.summary(idata, var_names=["alpha", "tau_s", "sigma_t", "alpha_nb", "beta"]))
